# Python Bug Detection Using CodeBERT on the BugInPy Dataset

**PhD topic:** Large Language Models for Automated Program Debugging
**Experiment 1:** Binary classification of Python code units — `1 = buggy`, `0 = non-buggy` — by fine-tuning `microsoft/codebert-base` on BugInPy.

---

## How to use this notebook

* This notebook is designed for **Google Colab** (`Runtime → Change runtime type → GPU`, e.g. T4).
* Run it **cell by cell**. Do **not** use `Runtime → Run all`.
* Every section follows the same structure:
  1. **Explanation** — what the step does.
  2. **Scientific objective** — why the step matters for the validity of the experiment.
  3. **Code** — the cell(s) to execute.
  4. **Expected output** — what you should see.
  5. **Verification** — an automated check (`assert`) that fails loudly if something is wrong.
  6. **Interpretation** — how to read the result.
* No results in this notebook are pre-filled. All numbers come from your own execution.

## Experimental phases

| Phase | Title | Status |
|---|---|---|
| 1 | Environment setup | **available** |
| 2 | BugInPy download | pending |
| 3 | Dataset exploration | pending |
| 4 | Dataset construction (labeling strategy) | pending |
| 5 | Data preprocessing & tokenization | pending |
| 6 | Leakage-free dataset splitting | pending |
| 7 | Baselines (TF-IDF + LR / RF) | pending |
| 8 | CodeBERT fine-tuning | pending |
| 9 | Evaluation | pending |
| 10 | Scientific analysis & threats to validity | pending |

Phases are added to this notebook incrementally, after the results of the previous phase are confirmed.

---
# PHASE 1 — Environment Setup

**Goal of the phase:** build a known, documented and reproducible computational environment before touching any data.

Sections:
* 1.1 Python version and global reproducibility flags
* 1.2 GPU availability and hardware information
* 1.3 Installation of required libraries
* 1.4 Imports and library versions
* 1.5 Random seeds and determinism
* 1.6 (Optional) Google Drive mounting and experiment directory layout
* 1.7 Experiment configuration and environment record
* 1.8 Phase 1 verification summary

## 1.1 Python version and global reproducibility flags

**Explanation.** We print the Python interpreter version and platform, and set environment variables that must exist *before* PyTorch initialises CUDA.

**Scientific objective.** The software stack is part of the experimental setup. Recording it allows other researchers (and reviewers) to reproduce the run. `CUBLAS_WORKSPACE_CONFIG` is required by PyTorch for deterministic cuBLAS operations; `PYTHONHASHSEED` fixes hashing in this process (e.g. ordering of `set` iteration in child processes).

**Expected output.** A Python version ≥ 3.9 (Colab currently ships Python 3.10–3.12), the platform string, and confirmation that the variables are set.

In [ ]:
# ---- 1.1 Python version & reproducibility flags ----
import os, sys, platform

# Must be set BEFORE torch initialises CUDA (i.e. before the first `import torch`).
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = "42"
os.environ["TOKENIZERS_PARALLELISM"] = "false"   # avoids fork warnings from HF tokenizers

IN_COLAB = "google.colab" in sys.modules

print("Python version :", sys.version.split()[0])
print("Implementation :", platform.python_implementation())
print("Platform       :", platform.platform())
print("Running in Colab:", IN_COLAB)
print("CUBLAS_WORKSPACE_CONFIG =", os.environ["CUBLAS_WORKSPACE_CONFIG"])

# ---- Verification ----
assert sys.version_info >= (3, 9), "Python >= 3.9 is required."
if not IN_COLAB:
    print("WARNING: this notebook is designed for Google Colab; some cells (Drive) will be skipped.")
print("\n[OK] 1.1 passed")

**Interpretation.** If `Running in Colab` is `False`, you are not in Colab (e.g. local Jupyter); the notebook still works but Drive integration is skipped. Note the Python version — it will be stored automatically in the environment record (1.7).

## 1.2 GPU availability and hardware information

**Explanation.** We query the NVIDIA driver (`nvidia-smi`) and PyTorch's view of CUDA, and record CPU / RAM / disk.

**Scientific objective.** Training time, maximum batch size and even small numerical differences depend on the hardware. Fine-tuning CodeBERT (≈125 M parameters) on CPU is impractically slow, so a GPU is strongly recommended for Phase 8. Baselines (Phase 7) can run on CPU.

**Expected output.**
* With a GPU runtime: an `nvidia-smi` table (typically *Tesla T4, 15 GB*; on Colab Pro possibly L4 / A100), `CUDA available: True`, the GPU name and memory.
* Without a GPU: `CUDA available: False` and a warning. In that case go to **Runtime → Change runtime type → T4 GPU**, then restart from cell 1.1.

In [ ]:
# ---- 1.2 GPU & hardware information ----
import subprocess, shutil

def run(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout.strip()
    except Exception as e:
        return f"<failed: {e}>"

print("=== nvidia-smi ===")
print(run("nvidia-smi") or "nvidia-smi not available (no GPU attached).")

import torch
cuda_ok = torch.cuda.is_available()
print("\n=== PyTorch view ===")
print("torch version   :", torch.__version__)
print("CUDA available  :", cuda_ok)
if cuda_ok:
    props = torch.cuda.get_device_properties(0)
    print("CUDA (torch)    :", torch.version.cuda)
    print("cuDNN           :", torch.backends.cudnn.version())
    print("GPU count       :", torch.cuda.device_count())
    print("GPU name        :", props.name)
    print(f"GPU memory      : {props.total_memory / 1024**3:.1f} GB")
    print(f"Compute capab.  : {props.major}.{props.minor}")
    print("bf16 supported  :", torch.cuda.is_bf16_supported())

print("\n=== CPU / RAM / Disk ===")
print("CPU             :", run("lscpu | grep 'Model name' | sed 's/Model name:\\s*//'") or platform.processor())
print("Logical cores   :", os.cpu_count())
print("RAM             :", run("free -h | awk '/Mem:/ {print $2\" total, \"$7\" available\"}'"))
total, used, free = shutil.disk_usage("/")
print(f"Disk (/)        : {total/1024**3:.0f} GB total, {free/1024**3:.0f} GB free")

DEVICE = torch.device("cuda" if cuda_ok else "cpu")
print("\nSelected DEVICE :", DEVICE)

# ---- Verification ----
if not cuda_ok:
    print("\nWARNING: No GPU detected. Phases 1-7 can run on CPU, but Phase 8 (CodeBERT) "
          "will be extremely slow. Switch to a GPU runtime before Phase 8.")
else:
    x = torch.randn(1024, 1024, device=DEVICE)
    assert torch.isfinite(x @ x).all(), "GPU matmul sanity check failed."
    print("GPU matmul sanity check: OK")
print("\n[OK] 1.2 passed")

**Interpretation.** Note the GPU model and memory: with a 15 GB T4, CodeBERT with `max_length=512` fits comfortably with batch size 8–16 (mixed precision). The GPU name is stored in the environment record, and will be reported together with training time in Phase 8.

## 1.3 Installation of required libraries

**Explanation.** Colab already provides `torch`, `numpy`, `pandas`, `scikit-learn` and `matplotlib`. We install/upgrade the Hugging Face stack and a few utilities. We deliberately **do not reinstall PyTorch**, to keep the CUDA build that matches Colab's driver.

| Library | Role in the experiment |
|---|---|
| `transformers` | CodeBERT model + tokenizer, training utilities |
| `datasets` | efficient tokenized dataset storage |
| `accelerate` | required by `transformers` Trainer / mixed precision |
| `evaluate` | standard metric implementations (cross-check) |
| `scikit-learn` | TF-IDF baselines, metrics, splitting utilities |
| `seaborn`, `matplotlib` | figures |
| `tqdm` | progress bars |
| `GitPython` | inspecting BugInPy git metadata (Phase 2–4) |

**Scientific objective.** A controlled and recorded dependency set. Exact installed versions are logged in 1.4 and saved in 1.7.

**Expected output.** pip progress (quiet mode prints little). Possible *dependency-resolver warnings* about unrelated Colab packages are normally harmless. If pip reports that a **restart is required**, use `Runtime → Restart session`, then re-run from **1.1** (skip 1.3).

In [ ]:
# ---- 1.3 Install libraries (run once per Colab session) ----
%pip install -q -U "transformers>=4.44,<5" "datasets>=2.20" "accelerate>=0.33" "evaluate>=0.4" \
                   "scikit-learn>=1.3" seaborn tqdm GitPython
print("\n[OK] 1.3 installation finished — check above for errors (warnings are usually fine).")

**Verification.** The next cell (1.4) imports every library and prints its version; an `ImportError` there means an installation failed.

**Interpretation.** Version ranges (not exact pins) are used because Colab images change over time; the *exact* versions actually used are what matters for reproducibility, and those are recorded.

## 1.4 Imports and library versions

**Explanation.** Import every dependency used across all phases and print the versions.

**Scientific objective.** Fail early if anything is missing, and document the precise software versions.

**Expected output.** A table of library versions, e.g. `transformers 4.4x.x`, `torch 2.x.x+cu12x`, and `[OK] 1.4 passed`.

In [ ]:
# ---- 1.4 Imports ----
import json, time, random, re, ast, hashlib, textwrap, warnings, datetime, logging
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import sklearn
import transformers
import datasets
import accelerate
import evaluate
import git  # GitPython

from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 120)

LIB_VERSIONS = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
    "evaluate": evaluate.__version__,
    "scikit-learn": sklearn.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
    "GitPython": git.__version__,
}
print(pd.Series(LIB_VERSIONS, name="version").to_string())

# ---- Verification ----
assert int(transformers.__version__.split(".")[0]) == 4, "Expected transformers 4.x"
print("\n[OK] 1.4 passed")

**Interpretation.** These versions are saved to `environment.json` in 1.7 and must be reported in the methodology section of the paper/thesis.

## 1.5 Random seeds and determinism

**Explanation.** Fix the seeds of Python `random`, NumPy and PyTorch (CPU + CUDA), and enable deterministic algorithms.

**Scientific objective.** Fine-tuning results vary with weight initialisation of the classification head, data shuffling and dropout. Fixing seeds makes a single run repeatable. *Note:* full bit-level determinism on GPU is not always guaranteed (some CUDA kernels are non-deterministic); for the final thesis results we will therefore also report the **mean ± std over several seeds** (planned in Phase 8/9).

**Expected output.** `Seed set to 42`, followed by two identical random draws (re-seeding test) and `[OK] 1.5 passed`.

In [ ]:
# ---- 1.5 Seeds ----
SEED = 42

def set_seed(seed: int = SEED, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    transformers.set_seed(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        # warn_only=True: report (not crash on) ops without a deterministic implementation
        torch.use_deterministic_algorithms(True, warn_only=True)
    print(f"Seed set to {seed} (deterministic={deterministic})")

set_seed(SEED)

# ---- Verification: re-seeding must reproduce the same numbers ----
def draw():
    return (random.random(), float(np.random.rand()), float(torch.rand(1)))
set_seed(SEED); a = draw()
set_seed(SEED); b = draw()
print("draw 1:", a)
print("draw 2:", b)
assert a == b, "Seeding is not reproducible!"
set_seed(SEED)
print("\n[OK] 1.5 passed")

**Interpretation.** Identical draws confirm that every random generator used by the pipeline is controlled by `SEED`. The seed value is recorded in the configuration (1.7).

## 1.6 (Optional) Google Drive mounting and directory layout

**Explanation.** Colab's local disk is erased when the session ends. Mounting Google Drive lets us persist the processed dataset, models, checkpoints, logs, metrics and figures. Set `USE_DRIVE = False` to work only on the temporary local disk.

When the cell runs, Colab will open an authorisation pop-up — accept access to your Drive.

**Directory layout** (created under `MyDrive/PhD_Experiments/BugInPy_CodeBERT/`):

```
BugInPy_CodeBERT/
├── data/
│   ├── raw/          # untouched copy of BugInPy (Phase 2) — never modified
│   ├── interim/      # extracted code units before cleaning (Phase 4)
│   └── processed/    # final labelled dataset + split files (Phases 5-6)
├── models/
│   ├── baselines/    # TF-IDF + LR / RF (Phase 7)
│   └── codebert/     # final fine-tuned model + tokenizer (Phase 8)
├── checkpoints/      # intermediate training checkpoints
├── logs/             # training logs, environment & config records
├── results/
│   ├── metrics/      # CSV / JSON evaluation results
│   └── predictions/  # per-sample predictions (error analysis)
└── figures/          # plots (loss curves, ROC, confusion matrix ...)
```

**Scientific objective.** Persistent, organised storage of every artefact, which is required for reproducibility and for later phases of the PhD.

**Expected output.** `Mounted at /content/drive` (if `USE_DRIVE=True`) and a listing of the created folders.

In [ ]:
# ---- 1.6 Optional Google Drive + directory layout ----
USE_DRIVE = True          # <-- set to False to keep everything on the temporary Colab disk
EXPERIMENT_NAME = "BugInPy_CodeBERT"

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/PhD_Experiments") / EXPERIMENT_NAME
else:
    if USE_DRIVE and not IN_COLAB:
        print("Not running in Colab: Drive unavailable, using local directory.")
    BASE_DIR = (Path("/content") if IN_COLAB else Path.cwd()) / EXPERIMENT_NAME

# Fast local working dir (git clone etc. is much faster on local disk than on Drive)
WORK_DIR = (Path("/content") if IN_COLAB else Path.cwd()) / "work"

PATHS = {
    "base":        BASE_DIR,
    "raw":         BASE_DIR / "data" / "raw",
    "interim":     BASE_DIR / "data" / "interim",
    "processed":   BASE_DIR / "data" / "processed",
    "baselines":   BASE_DIR / "models" / "baselines",
    "codebert":    BASE_DIR / "models" / "codebert",
    "checkpoints": BASE_DIR / "checkpoints",
    "logs":        BASE_DIR / "logs",
    "metrics":     BASE_DIR / "results" / "metrics",
    "predictions": BASE_DIR / "results" / "predictions",
    "figures":     BASE_DIR / "figures",
    "work":        WORK_DIR,
}
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

for k, p in PATHS.items():
    print(f"{k:<12} -> {p}")

# ---- Verification: every directory exists and is writable ----
for k, p in PATHS.items():
    assert p.is_dir(), f"Missing dir: {p}"
    probe = p / ".write_test"
    probe.write_text("ok"); probe.unlink()
print("\nStorage persistent across sessions:", USE_DRIVE and IN_COLAB)
print("[OK] 1.6 passed")

**Interpretation.** If `Storage persistent across sessions: False`, all outputs will be lost when the Colab runtime is recycled — acceptable for testing, not for final experiments. `WORK_DIR` (local disk) is used for heavy temporary operations; final artefacts are always copied to `BASE_DIR`.

## 1.7 Experiment configuration and environment record

**Explanation.** A single `CONFIG` dictionary centralises every experimental choice (seed, model, hyperparameters, split ratios). Later phases read from it — nothing is hard-coded elsewhere. The configuration and full environment description are saved as JSON with a unique **run ID**.

**Scientific objective.** Research reproducibility: seed, dataset version, model name, hyperparameters and hardware are all recorded. Fields marked `None` are filled in by later phases (e.g. the exact BugInPy commit hash in Phase 2, split identifiers in Phase 6, training time in Phase 8).

**Expected output.** The printed configuration and the paths of `config_<RUN_ID>.json` and `environment_<RUN_ID>.json` in `logs/`.

In [ ]:
# ---- 1.7 Configuration & environment record ----
RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

CONFIG = {
    "run_id": RUN_ID,
    "experiment": EXPERIMENT_NAME,
    "seed": SEED,
    # --- dataset (filled in Phase 2) ---
    "dataset": {
        "name": "BugInPy",
        "repository": "https://github.com/soarsmu/BugsInPy",
        "commit": None,          # exact commit hash recorded in Phase 2
    },
    # --- model ---
    "model_name": "microsoft/codebert-base",
    "num_labels": 2,
    "label_map": {"0": "non-buggy", "1": "buggy"},
    # --- training hyperparameters (Phase 8) ---
    "hyperparameters": {
        "optimizer": "AdamW",
        "learning_rate": 2e-5,
        "batch_size": 16 if cuda_ok else 8,
        "num_epochs": 5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "max_seq_length": 512,
        "early_stopping_patience": 2,
        "early_stopping_metric": "eval_f1",
        "loss": "CrossEntropyLoss",
        "fp16": bool(cuda_ok),
    },
    # --- splitting (Phase 6) ---
    "split": {
        "strategy": "project-level (grouped)",
        "ratios": {"train": 0.70, "val": 0.15, "test": 0.15},
        "split_ids_file": None,  # filled in Phase 6
    },
    "paths": {k: str(v) for k, v in PATHS.items()},
}

ENVIRONMENT = {
    "run_id": RUN_ID,
    "timestamp": datetime.datetime.now().isoformat(),
    "in_colab": IN_COLAB,
    "platform": platform.platform(),
    "libraries": LIB_VERSIONS,
    "hardware": {
        "device": str(DEVICE),
        "gpu_name": torch.cuda.get_device_name(0) if cuda_ok else None,
        "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if cuda_ok else None,
        "cuda_version": torch.version.cuda if cuda_ok else None,
        "cudnn_version": torch.backends.cudnn.version() if cuda_ok else None,
        "cpu_count": os.cpu_count(),
        "cpu_model": run("lscpu | grep 'Model name' | sed 's/Model name:\\s*//'"),
    },
    "pip_freeze_file": f"pip_freeze_{RUN_ID}.txt",
}

def save_json(obj, path):
    path = Path(path)
    path.write_text(json.dumps(obj, indent=2, default=str))
    return path

cfg_path = save_json(CONFIG, PATHS["logs"] / f"config_{RUN_ID}.json")
env_path = save_json(ENVIRONMENT, PATHS["logs"] / f"environment_{RUN_ID}.json")
(PATHS["logs"] / ENVIRONMENT["pip_freeze_file"]).write_text(run(f"{sys.executable} -m pip freeze"))

print(json.dumps(CONFIG, indent=2)[:2500])
print("\nSaved:", cfg_path)
print("Saved:", env_path)
print("Saved:", PATHS["logs"] / ENVIRONMENT["pip_freeze_file"])

# ---- Verification: files can be read back and match ----
assert json.loads(cfg_path.read_text())["seed"] == SEED
assert json.loads(env_path.read_text())["libraries"]["transformers"] == transformers.__version__
print("\n[OK] 1.7 passed")

**Interpretation.** `RUN_ID` uniquely identifies this experimental run; every file produced later will carry it. The `pip_freeze` file is the most precise record of the software environment and can be used to rebuild it exactly.

> **Note on hyperparameters.** The values above follow your specification (AdamW, lr = 2e-5, batch 8/16, 5 epochs, early stopping, cross-entropy). `weight_decay=0.01`, `warmup_ratio=0.1` and `max_seq_length=512` are standard defaults for CodeBERT fine-tuning; they will be discussed again in Phase 5 (sequence length) and Phase 8 before training.

## 1.8 Phase 1 verification summary

**Explanation.** A final checklist of the whole phase.

**Expected output.** A table where every row says `PASS` (the GPU row may say `WARN` on a CPU runtime).

In [ ]:
# ---- 1.8 Phase 1 summary ----
checks = {
    "Python >= 3.9":                 "PASS" if sys.version_info >= (3, 9) else "FAIL",
    "GPU available":                 "PASS" if cuda_ok else "WARN (CPU only)",
    "transformers importable":       "PASS",
    "Seeds reproducible":            "PASS",
    "Output directories writable":   "PASS",
    "Persistent storage (Drive)":    "PASS" if (USE_DRIVE and IN_COLAB) else "WARN (temporary disk)",
    "Config + environment saved":    "PASS" if cfg_path.exists() and env_path.exists() else "FAIL",
}
summary = pd.DataFrame({"check": list(checks), "status": list(checks.values())})
print(summary.to_string(index=False))
print(f"\nRUN_ID = {RUN_ID}   DEVICE = {DEVICE}   BASE_DIR = {BASE_DIR}")
assert "FAIL" not in checks.values(), "Phase 1 has failing checks — fix them before continuing."
print("\n[OK] PHASE 1 COMPLETE — report these outputs before moving to Phase 2.")

**Interpretation.** Phase 1 is complete when no row is `FAIL`. Please report back:
1. The output of **1.2** (GPU name & memory),
2. The library versions from **1.4**,
3. The summary table of **1.8**,
4. Any warning or error message.

➡️ **PHASE 2 (BugInPy download) will be added after these results are confirmed.**